# 06-window-functions

06-window-functions/01-window-functions.py
==========================================
Covers 10 window function patterns using the shared HR/ops seed dataset.

Sections
--------
 1. Window spec basics: row_number / rank / dense_rank + NULL sort order
 2. ROW_NUMBER for top-N per group
 3. NTILE — salary quartiles
 4. LAG and LEAD on salary history
 5. Running SUM (cumulative salary spend)
 6. ROWS vs RANGE frame difference
 7. FIRST_VALUE and LAST_VALUE (unbounded frame gotcha)
 8. Rolling 3-period window aggregate on purchase orders
 9. Percent of partition total
10. Dense rank on performance rating per period

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("06-window-functions")
dfs = register_views(spark)
emp = dfs["employees"]
sal = dfs["salary_history"]
pr  = dfs["performance_reviews"]
po  = dfs["purchase_orders"]

## SECTION 1 — Window spec basics: row_number / rank / dense_rank

In [ ]:
# Default .desc() sorts NULLs LAST in PySpark; make it explicit.
# Use desc_nulls_last() vs desc_nulls_first() to control NULL placement.

print("\n=== SECTION 1: row_number / rank / dense_rank per dept (salary desc) ===")
print("NOTE: emp 10, 15 have salary=None — sorted LAST with desc_nulls_last()")

w_dept = Window.partitionBy("dept_id").orderBy(F.col("salary").desc_nulls_last())

emp.withColumn("rn",        F.row_number().over(w_dept)) \
   .withColumn("rnk",       F.rank().over(w_dept)) \
   .withColumn("dense_rnk", F.dense_rank().over(w_dept)) \
   .select("emp_id", "first_name", "dept_id", "salary", "rn", "rnk", "dense_rnk") \
   .filter(F.col("rn") <= 3) \
   .orderBy("dept_id", "rn") \
   .show(30)

# Flip to desc_nulls_first() to push NULLs to the top
print("--- Same spec but desc_nulls_first() → NULLs bubble to rank 1 ---")
w_dept_nf = Window.partitionBy("dept_id").orderBy(F.col("salary").desc_nulls_first())
emp.withColumn("rn", F.row_number().over(w_dept_nf)) \
   .select("emp_id", "first_name", "dept_id", "salary", "rn") \
   .filter(F.col("emp_id").isin(10, 15)) \
   .show()

## SECTION 2 — ROW_NUMBER for top-N per group

In [ ]:
# NULLs excluded from top positions by using desc_nulls_last().

print("\n=== SECTION 2: top-2 earners per department ===")
print("DATA FLAW: emp 10, 15 (salary=None) land at bottom — won't appear in top-2")

w_top = Window.partitionBy("dept_id").orderBy(F.col("salary").desc_nulls_last())
top2_per_dept = (
    emp.withColumn("rn", F.row_number().over(w_top))
       .filter(F.col("rn") <= 2)
)
top2_per_dept.select("dept_id", "emp_id", "first_name", "salary", "rn") \
             .orderBy("dept_id", "rn") \
             .show(30)

## SECTION 3 — NTILE — salary quartiles

In [ ]:
# ntile(4) assigns rows to 4 buckets based on ORDER BY.
# NULLs sorted last — emp 10, 15 land in quartile 4.

print("\n=== SECTION 3: salary quartiles via ntile(4) ===")
print("DATA FLAW: emp 10, 15 (salary=None) sorted last → quartile 4")
print("DATA FLAW: emp 19 (salary=0.0) correctly lands in quartile 1")

w_salary = Window.orderBy(F.col("salary").asc_nulls_last())
emp.withColumn("quartile", F.ntile(4).over(w_salary)) \
   .select("emp_id", "first_name", "salary", "quartile") \
   .orderBy("salary_nulls_last", ascending=True) \
   .show(10)

## SECTION 4 — LAG and LEAD on salary history

In [ ]:
# lag(col, 1) → previous row value; NULL at partition start (first row per employee).
# lead(col, 1) → next row value; NULL at partition end.
# salary_history rows 15,16 are exact duplicates for emp 5 on 2022-04-01.

print("\n=== SECTION 4: LAG / LEAD — salary changes over time ===")
print("DATA FLAW: rows 15,16 are exact dups for emp 5 / 2022-04-01 → delta=0.0 on that row")
print("NOTE: prev_salary is NULL for the first (earliest) salary row per employee")

w_emp = Window.partitionBy("emp_id").orderBy("effective_date")
sal.withColumn("prev_salary",   F.lag("salary_after", 1).over(w_emp)) \
   .withColumn("next_salary",   F.lead("salary_after", 1).over(w_emp)) \
   .withColumn("salary_delta",  F.col("salary_after") - F.col("prev_salary")) \
   .select("emp_id", "effective_date", "salary_after", "prev_salary", "next_salary", "salary_delta") \
   .show(15)

## SECTION 5 — Running SUM (cumulative salary spend per employee)

In [ ]:
# rowsBetween(unboundedPreceding, currentRow) → classic cumulative sum.

print("\n=== SECTION 5: running cumulative sum of salary_after per employee ===")

w_run = (
    Window.partitionBy("emp_id")
          .orderBy("effective_date")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
sal.withColumn("running_total", F.sum("salary_after").over(w_run)) \
   .select("emp_id", "effective_date", "salary_after", "running_total") \
   .show(15)

## SECTION 6 — ROWS vs RANGE frame difference

In [ ]:
# ROWS counts physical rows; RANGE uses the ORDER BY value range.
# When two rows share the same salary (ties), RANGE groups them
# together — both get the sum including each other.

print("\n=== SECTION 6: ROWS vs RANGE frame ===")
print("Filtered to dept_id=2 where tied salaries exist.")
print("RANGE lumps tied values → range_sum differs from rows_sum for ties.")

w_rows  = Window.partitionBy("dept_id").orderBy("salary").rowsBetween(Window.unboundedPreceding, Window.currentRow)
w_range = Window.partitionBy("dept_id").orderBy("salary").rangeBetween(Window.unboundedPreceding, Window.currentRow)

emp.filter(F.col("dept_id") == 2) \
   .withColumn("rows_sum",  F.sum("salary").over(w_rows)) \
   .withColumn("range_sum", F.sum("salary").over(w_range)) \
   .select("emp_id", "first_name", "salary", "rows_sum", "range_sum") \
   .orderBy("salary") \
   .show()

## SECTION 7 — FIRST_VALUE and LAST_VALUE

In [ ]:
# IMPORTANT: LAST_VALUE with default frame (unboundedPreceding → currentRow)
# always returns the current row value — not the last.
# Fix: use rowsBetween(unboundedPreceding, unboundedFollowing).

print("\n=== SECTION 7: FIRST_VALUE and LAST_VALUE (unbounded frame) ===")
print("GOTCHA: without unboundedFollowing, last_value = current row (not last row)")

w_unbounded = (
    Window.partitionBy("emp_id")
          .orderBy("effective_date")
          .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)
sal.withColumn("first_salary", F.first("salary_after").over(w_unbounded)) \
   .withColumn("last_salary",  F.last("salary_after").over(w_unbounded)) \
   .select("emp_id", "effective_date", "salary_after", "first_salary", "last_salary") \
   .show(15)

## SECTION 8 — Rolling 3-period window aggregate on purchase orders

In [ ]:
# Aggregate to monthly totals first, then apply a 3-month rolling sum
# using rowsBetween(-2, 0) — current row + 2 preceding rows.

print("\n=== SECTION 8: rolling 3-month spend on purchase orders ===")

monthly = (
    po.groupBy(F.date_format("order_date", "yyyy-MM").alias("month"))
      .agg(F.sum("amount").alias("monthly_total"))
      .orderBy("month")
)

w_roll = Window.orderBy("month").rowsBetween(-2, 0)
monthly.withColumn("rolling_3m_total", F.sum("monthly_total").over(w_roll)) \
       .show(20)

## SECTION 9 — Percent of partition total

In [ ]:
# Partition-level aggregation without ORDER BY or frame → returns
# the same value for all rows in the partition (group total).
# Dividing each row's salary by that total gives the % share.

print("\n=== SECTION 9: each employee's % share of department salary ===")
print("DATA FLAW: emp 10, 15 (salary=None) → pct_of_dept=None (NULL propagates)")

w_dept_total = Window.partitionBy("dept_id")
emp.withColumn("dept_total_salary",
               F.sum("salary").over(w_dept_total)) \
   .withColumn("pct_of_dept",
               F.round(F.col("salary") / F.col("dept_total_salary") * 100, 1)) \
   .select("emp_id", "first_name", "dept_id", "salary", "dept_total_salary", "pct_of_dept") \
   .filter(F.col("dept_id") == 1) \
   .orderBy("salary") \
   .show()

## SECTION 10 — Dense rank on performance rating per review period

In [ ]:
# desc_nulls_last() keeps NULL-rating employees at the bottom.
# dense_rank avoids gaps when multiple employees share the same rating.

print("\n=== SECTION 10: dense_rank on performance rating per review period ===")
print("DATA FLAW: some rating values are NULL → sorted last via desc_nulls_last()")

w_rating = Window.partitionBy("review_period").orderBy(F.col("rating").desc_nulls_last())
pr.withColumn("rating_rank", F.dense_rank().over(w_rating)) \
  .select("emp_id", "review_period", "rating", "rating_rank") \
  .orderBy("review_period", "rating_rank") \
  .show(30)

## stop_and_wait(spark)